# Temporal Detection Evaluation

Evaluate cat and person tracks using temporal IoU, precision, recall and F1.

## Configure notebook

In [ ]:
from nb_utils import setup_notebook

PROJECT_ROOT = setup_notebook()

In [ ]:
from collections import defaultdict
from collections.abc import Iterable
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.optimize import linear_sum_assignment

import batch_run
from config import SETTINGS_PATH_OVERRIDE, settings
from nb_utils import (
    PERSON_LABEL_ID,
    VideoInfo,
    annotation_intervals,
    is_video_annotated,
    load_annotations,
)
from utils import get_video_paths

In [ ]:
# define constants
TIOU_THRESHOLDS = np.arange(0.5, 1.0, 0.1).round(1)
PRIMARY_TIOU_THRESHOLD = TIOU_THRESHOLDS[0]
N_TRIALS = 4

In [ ]:
# define paths
OUTPUT_DIR = PROJECT_ROOT / "object_clips"
METADATA_DIR = OUTPUT_DIR / "metadata"
LABELS_PATH = PROJECT_ROOT / "datasets" / "behaviour_dataset" / "behaviour_labels.json"

## Load target videos

In [ ]:
# all processed or annotated videos should be a subset of target videos
video_paths = get_video_paths(mock_inputs_long=True)
video_paths.sort(key=lambda p: p.stat().st_size)
video_paths = video_paths[:10]
video_info = {p.name: VideoInfo(p) for p in video_paths}

## Define functions

In [ ]:
# define functions
def temporal_iou(a: dict, b: dict) -> float:
    """Calculate temporal intersection over union for inclusive frame intervals"""
    intersection = max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]) + 1)
    union = (a["end"] - a["start"] + 1) + (b["end"] - b["start"] + 1) - intersection
    return intersection / union


def match_temporal_intervals(
    ground_truth: list[dict],
    predictions: list[dict],
    thresholds: Iterable[float],
    video_names: Iterable[str],
) -> dict:
    """Match intervals one-to-one at each temporal IoU threshold"""
    threshold_values = tuple(thresholds)
    matches_by_threshold = {t: [] for t in threshold_values}
    for video_name in video_names:
        for class_name in ("cat", "person"):
            truth = [
                r
                for r in ground_truth
                if r["video"] == video_name and r["class"] == class_name
            ]
            tracks = [
                r
                for r in predictions
                if r["video"] == video_name and r["class"] == class_name
            ]
            if truth and tracks:
                overlaps = np.array(
                    [[temporal_iou(g, p) for p in tracks] for g in truth]
                )
                match_count = min(overlaps.shape)
                for threshold in threshold_values:
                    valid_matches = overlaps >= threshold
                    # prioritize the number of valid matches before their overlap
                    scores = np.where(valid_matches, match_count + 1 + overlaps, 0)
                    rows, columns = linear_sum_assignment(scores, maximize=True)
                    matches_by_threshold[threshold].extend(
                        (truth[i], tracks[j])
                        for i, j in zip(rows, columns)
                        if valid_matches[i, j]
                    )

    return matches_by_threshold


def evaluate_temporal_predictions(
    ground_truth: list[dict],
    predictions: list[dict],
    thresholds: Iterable[float],
    video_names: Iterable[str],
) -> tuple[pd.DataFrame, dict]:
    """Calculate class and pooled temporal detection metrics"""
    matches_by_threshold = match_temporal_intervals(
        ground_truth, predictions, thresholds, video_names
    )
    metrics = []

    for threshold, threshold_matches in matches_by_threshold.items():
        for class_name in ("cat", "person", "all"):
            truth = [
                r
                for r in ground_truth
                if class_name == "all" or r["class"] == class_name
            ]
            tracks = [
                r
                for r in predictions
                if class_name == "all" or r["class"] == class_name
            ]
            true_positives = sum(
                class_name == "all" or g["class"] == class_name
                for g, _ in threshold_matches
            )
            metrics.append(
                {
                    "tIoU": threshold,
                    "class": class_name,
                    "ground-truth intervals": len(truth),
                    "predicted tracks": len(tracks),
                    "TP": true_positives,
                }
            )

    metrics = pd.DataFrame(metrics).set_index(["tIoU", "class"])
    metrics["FP"] = metrics["predicted tracks"] - metrics["TP"]
    metrics["FN"] = metrics["ground-truth intervals"] - metrics["TP"]
    metrics["precision"] = metrics["TP"].div(metrics["predicted tracks"]).fillna(0)
    metrics["recall"] = metrics["TP"].div(metrics["ground-truth intervals"]).fillna(0)
    metrics["F1"] = (
        (2 * metrics["TP"])
        .div(2 * metrics["TP"] + metrics["FP"] + metrics["FN"])
        .fillna(0)
    )
    return metrics, matches_by_threshold


def build_ground_truth(
    annotations: list[dict],
    ann_video_info: dict[str, VideoInfo],
    stale_duration: float,
) -> list[dict]:
    """Merge annotation windows into class-specific presence intervals"""
    # collect class-specific annotation windows
    windows = defaultdict(list)
    for video_name, video in ann_video_info.items():
        intervals = annotation_intervals(
            annotations, video.hashes, video_name=video_name
        )
        for cat_id, start, end, _ in intervals:
            class_name = "person" if cat_id == PERSON_LABEL_ID else "cat"
            identity = cat_id if class_name == "cat" else "person"
            windows[(video_name, class_name, identity)].append((start, end))

    # merge windows when the same object could remain in the track's stale state
    ground_truth = []
    for (name, class_name, _), intervals in windows.items():
        merged = []
        for start, end in sorted(intervals):
            gap = start - merged[-1][1] - 1 if merged else None
            if merged and gap < stale_duration * ann_video_info[name].fps:
                merged[-1] = (merged[-1][0], max(merged[-1][1], end))
            else:
                merged.append((start, end))
        ground_truth.extend(
            {"video": name, "class": class_name, "start": start, "end": end}
            for start, end in merged
        )

    return ground_truth


def load_prediction_intervals(
    tracks: list[dict], video_info: dict[str, VideoInfo]
) -> list[dict]:
    """Convert exported tracks into source-frame intervals"""
    video_hash_indices = {
        name: {frame_hash: index for index, frame_hash in enumerate(video.hashes)}
        for name, video in video_info.items()
    }
    predictions = []

    for track in tracks:
        name = Path(track["video_path"]).name
        if name in video_info:
            hash_indices = video_hash_indices[name]
            frame_indices = [
                hash_indices[frame_hash]
                for frame_hash in track["frame_hashes"]
                if frame_hash in hash_indices
            ]
            if frame_indices:
                predictions.append(
                    {
                        "video": name,
                        "class": track["object_name"],
                        "start": min(frame_indices),
                        "end": max(frame_indices),
                    }
                )

    return predictions


def evaluate_trial(
    trial: optuna.trial.Trial,
    clip_paths: list[Path],
    ground_truth: list[dict],
    video_info: dict[str, VideoInfo],
) -> float:
    """Run and evaluate one confidence-threshold configuration"""

    conf = trial.suggest_float("CONF", 0.1, 0.35, step=0.05)
    baseline_conf = float(settings.CONF)
    processed_video_names = []
    elapsed_start = perf_counter()

    # create a per-trial settings file for the child processes
    try:
        SETTINGS_PATH_OVERRIDE.write_text(
            f'CONF = {conf!r}\nLOG_LEVEL = "ERROR"\nEXCLUDED_OBJECTS = ""\n'
        )

        # clear outputs before processing this configuration
        for index, video_path in enumerate(clip_paths):
            batch_run.main(
                [video_path],
                clear_output=index == 0,
            )
            processed_video_names.append(video_path.name)

            # evaluate all clips processed so far
            processed_names = set(processed_video_names)
            trial_ground_truth = [
                i for i in ground_truth if i["video"] in processed_names
            ]
            trial_tracks = [
                joblib.load(path) for path in METADATA_DIR.glob("track-*.joblib")
            ]
            predictions = load_prediction_intervals(trial_tracks, video_info)
            trial_metrics, _ = evaluate_temporal_predictions(
                trial_ground_truth,
                predictions,
                TIOU_THRESHOLDS,
                processed_video_names,
            )
            pooled_metrics = trial_metrics.xs("all", level="class")
            metrics_by_tiou = [
                {
                    "tIoU": float(threshold),
                    "precision": float(row["precision"]),
                    "recall": float(row["recall"]),
                    "F1": float(row["F1"]),
                }
                for threshold, row in pooled_metrics.iterrows()
            ]
            mean_pooled_f1 = float(pooled_metrics["F1"].mean())
            trial.report(mean_pooled_f1, step=index)
            trial.set_user_attr("metrics_by_tiou", metrics_by_tiou)
            trial.set_user_attr("processed_video_names", processed_video_names.copy())
            trial.set_user_attr("mean_pooled_f1", mean_pooled_f1)

        penalty = 0.01 * abs(conf - baseline_conf) / 0.05
        objective_score = mean_pooled_f1 - penalty
        trial.set_user_attr("baseline_penalty", penalty)
        return objective_score
    finally:
        SETTINGS_PATH_OVERRIDE.unlink(missing_ok=True)
        trial.set_user_attr("elapsed_seconds", perf_counter() - elapsed_start)

## Load ground truth

In [ ]:
# identify labelled target videos and report unlabelled videos
annotations = load_annotations(LABELS_PATH)
ann_video_info = {
    n: v for n, v in video_info.items() if is_video_annotated(v, annotations)
}
unlabelled_videos = set(video_info) - set(ann_video_info)
print(f"Unlabelled videos: {len(unlabelled_videos)}")
for video_name in sorted(unlabelled_videos):
    print(f"  {video_name}")

# merge annotation windows into presence intervals
ground_truth = build_ground_truth(annotations, ann_video_info, settings.TRACK_STALE_DUR)
print(f"Ground-truth intervals: {len(ground_truth)}")

## Run batch processing

In [ ]:
# apply evaluation-only settings to the child process
try:
    SETTINGS_PATH_OVERRIDE.write_text('LOG_LEVEL = "ERROR"\nEXCLUDED_OBJECTS = ""\n')
    return_code = batch_run.main(video_paths, clear_output=True)
finally:
    SETTINGS_PATH_OVERRIDE.unlink(missing_ok=True)

## Load predicted tracks

In [ ]:
# load exported tracks and report unprocessed target videos
tracks = [joblib.load(p) for p in METADATA_DIR.glob("track-*.joblib")]
proc_video_names = {
    n for t in tracks if (n := Path(t["video_path"]).name) in video_info
}
proc_video_info = {n: v for n, v in video_info.items() if n in proc_video_names}
unprocessed_videos = set(video_info) - set(proc_video_info)
print(f"Unprocessed videos: {len(unprocessed_videos)}")
for video_name in sorted(unprocessed_videos):
    print(f"  {video_name}")

# convert exported tracks into source-frame intervals
predictions = load_prediction_intervals(tracks, proc_video_info)

# report the number of predicted intervals
print(f"Predicted intervals: {len(predictions)}")

## Evaluate predictions

In [ ]:
# calculate metrics and matches for the configured thresholds
metrics, matches_by_threshold = evaluate_temporal_predictions(
    ground_truth, predictions, TIOU_THRESHOLDS, ann_video_info
)

# use the display threshold for latency and per-video diagnostics
matches = matches_by_threshold[PRIMARY_TIOU_THRESHOLD]

# store detection latency for each matched track
detection_latencies = pd.DataFrame(
    [
        {
            "video": g["video"],
            "class": p["class"],
            "latency_seconds": (p["start"] - g["start"]) / video_info[p["video"]].fps,
        }
        for g, p in matches
    ],
)

# show performance per class aggregated across threshold
display(metrics.round(2))
metrics.groupby(
    ["class", "ground-truth intervals", "predicted tracks"]
).mean().reset_index(["ground-truth intervals", "predicted tracks"]).round(2)

## Plot per-video timelines

In [ ]:
# find videos where intervals and matches disagree
video_names_with_errors = []
for video_name in ann_video_info:
    truth_count = sum(r["video"] == video_name for r in ground_truth)
    track_count = sum(r["video"] == video_name for r in predictions)
    match_count = sum(g["video"] == video_name for g, _ in matches)
    if not truth_count == track_count == match_count:
        video_names_with_errors.append(video_name)

# compare ground-truth and predicted intervals for videos with errors
if video_names_with_errors:
    fig, axes = plt.subplots(
        len(video_names_with_errors),
        1,
        figsize=(14, 2.5 * len(video_names_with_errors)),
        squeeze=False,
    )
    for ind, (axis, video_name) in enumerate(zip(axes[:, 0], video_names_with_errors)):
        for lane, class_name, intervals, color in (
            (0, "cat", ground_truth, "tab:green"),
            (1, "cat", predictions, "tab:blue"),
            (2, "person", ground_truth, "tab:orange"),
            (3, "person", predictions, "tab:red"),
        ):
            lane_intervals = [
                i
                for i in intervals
                if i["video"] == video_name and i["class"] == class_name
            ]
            for i, interval in enumerate(lane_intervals):
                overlapping_before = sum(
                    interval["start"] <= other["end"]
                    and other["start"] <= interval["end"]
                    for other in lane_intervals[:i]
                )
                overlaps_later = any(
                    interval["start"] <= other["end"]
                    and other["start"] <= interval["end"]
                    for other in lane_intervals[i + 1 :]
                )
                if overlapping_before or overlaps_later:
                    offset = 0.15 if overlapping_before % 2 else -0.15
                else:
                    offset = 0
                axis.plot(
                    [interval["start"], interval["end"]],
                    [lane + offset, lane + offset],
                    color=color,
                    linewidth=5,
                )
        axis.set_yticks(
            range(4), ["cat GT", "cat tracks", "person GT", "person tracks"]
        )
        axis.set_title(f"{ind}:   {video_name}")
        axis.set_xlabel("Source frame")
        axis.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    plt.show()
else:
    print("No videos with errors to plot")

## Plot detection latency

In [ ]:
# compare matched detection latency between object classes
fig, ax = plt.subplots(figsize=(15, 4))
sns.histplot(
    data=detection_latencies,
    x="latency_seconds",
    hue="class",
    stat="proportion",
    binwidth=0.2,
    common_norm=False,
    multiple="layer",
    element="step",
    fill=False,
    ax=ax,
)
ax.set_xlabel("Detection latency [s]")
ax.set_ylabel("Count")
ax.grid(axis="y")
fig.tight_layout()

## Optimisation

In [ ]:
# run a small end-to-end optimization on the selected evaluation videos
mwe_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=1),
    pruner=optuna.pruners.NopPruner(),
)
mwe_study.enqueue_trial({"CONF": float(settings.CONF)})
mwe_study.optimize(
    lambda trial: evaluate_trial(trial, video_paths, ground_truth, video_info),
    n_trials=N_TRIALS,
)

# compare the baseline configuration with the sampled alternatives
mwe_trial_results = pd.DataFrame(
    [
        {
            "trial": trial.number,
            "CONF": trial.params.get("CONF"),
            "state": trial.state.name,
            "mean pooled F1": trial.user_attrs.get("mean_pooled_f1"),
            "baseline penalty": trial.user_attrs.get("baseline_penalty"),
            "objective score": trial.value,
            "elapsed seconds": trial.user_attrs.get("elapsed_seconds"),
        }
        for trial in mwe_study.trials
    ]
)
mwe_metrics = pd.DataFrame(
    [
        {"trial": trial.number, "CONF": trial.params["CONF"], **row}
        for trial in mwe_study.trials
        for row in trial.user_attrs.get("metrics_by_tiou", [])
    ]
)
display(mwe_trial_results)
display(mwe_metrics)